# Qwen3.5-9B — Interactive Chat on Colab Free Tier (T4 GPU)

This notebook runs **Qwen3.5-9B**, quantized to GGUF (4-bit, `UD-Q4_K_XL`), fully offloaded to the GPU using `llama-cpp-python`. The quantized weights are ~6 GB, which comfortably fits on a free-tier T4 (15 GB VRAM) alongside a reasonable context window.

**A couple of things worth knowing before you start:**
- **Runtime**: Go to `Runtime → Change runtime type → T4 GPU` before running anything below.
- **Thinking model**: Qwen3.5 reasons inside `<think>...</think>` tags before giving its final answer by default. This notebook asks the model to skip that (non-thinking / instruct-style replies) and also strips any leftover `<think>` tags from what's displayed, so you get direct answers.
- **First run is slow**: compiling `llama-cpp-python` with CUDA support takes a few minutes, and downloading the ~6 GB model takes a few more. After that, cells re-run fast.
- Model card: [unsloth/Qwen3.5-9B-GGUF](https://huggingface.co/unsloth/Qwen3.5-9B-GGUF)


## 1. Confirm you have a GPU

If this errors or shows no GPU, go to `Runtime → Change runtime type` and select **T4 GPU**, then re-run.

In [1]:
!nvidia-smi

Mon Aug 24 08:04:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install `llama-cpp-python` with CUDA support

Built from source against Colab's CUDA toolkit so the model actually runs on the GPU (the plain `pip install llama-cpp-python` gives you a CPU-only build, which will be very slow for a 9B model). This step takes roughly 5–10 minutes — it only needs to run once per Colab session.

In [3]:
import os

os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"

!apt-get -qq update
!apt-get -qq install -y cmake ninja-build

!pip install -U pip

!pip install \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
    llama-cpp-python \
    huggingface_hub

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124


In [4]:
from llama_cpp import llama_cpp
print("CUDA-enabled build:", llama_cpp.llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


CUDA-enabled build: True


In [5]:
# Sanity check: confirm the build actually picked up CUDA support
from llama_cpp import llama_cpp
print("CUDA-enabled build:", llama_cpp.llama_supports_gpu_offload())

CUDA-enabled build: True


## 3. Download the quantized model

We use Unsloth's **Dynamic 2.0** GGUF quantization of Qwen3.5-9B at `UD-Q4_K_XL` (~6 GB) — a 4-bit quant that keeps key layers at higher precision for better quality than a naive 4-bit quant, while still being small enough for the free-tier T4 and Colab's disk/RAM limits.

If you hit disk or RAM issues, drop to a smaller quant such as `UD-Q3_K_XL` (~5 GB) or `UD-Q2_K_XL` (~4.1 GB) by changing `QUANT` below — quality drops a bit as you go smaller, but it'll run on tighter setups.

In [6]:
from huggingface_hub import hf_hub_download

REPO_ID = "unsloth/Qwen3.5-9B-GGUF"
QUANT = "UD-Q4_K_XL"   # ~5.97 GB. Alternatives: UD-Q3_K_XL (~5.05 GB), UD-Q2_K_XL (~4.12 GB)

from huggingface_hub import list_repo_files
matches = [f for f in list_repo_files(REPO_ID) if QUANT in f and f.endswith(".gguf")]
assert matches, f"No GGUF file found for quant '{QUANT}' in {REPO_ID}"
filename = matches[0]
print("Downloading:", filename)

model_path = hf_hub_download(repo_id=REPO_ID, filename=filename)
print("Saved to:", model_path)

Downloading: Qwen3.5-9B-UD-Q4_K_XL.gguf


Qwen3.5-9B-UD-Q4_K_XL.gguf: reconstructing file:   0%|          |  0.00B / 5.97GB            

Qwen3.5-9B-UD-Q4_K_XL.gguf: downloading bytes:           |  0.00B            

Saved to: /root/.cache/huggingface/hub/models--unsloth--Qwen3.5-9B-GGUF/snapshots/3885219b6810b007914f3a7950a8d1b469d598a5/Qwen3.5-9B-UD-Q4_K_XL.gguf


## 4. Load the model

- `n_gpu_layers=-1` offloads every layer to the GPU.
- `n_ctx=8192` is a comfortable context length for the free-tier T4. Qwen3.5 natively supports up to 262,144 tokens, but a larger `n_ctx` means a larger KV cache in VRAM — raise this only if you have headroom left (check with `!nvidia-smi` after loading) and lower it if you hit an out-of-memory error.

In [7]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,      # offload all layers to the T4
    n_ctx=8192,           # context window; raise/lower based on available VRAM
    n_batch=512,
    flash_attn=True,
    verbose=False,
)
print("Model loaded.")

Model loaded.


In [8]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
6049 MiB, 15360 MiB


In [24]:
!pip install -q fastapi uvicorn pyngrok

In [25]:
from fastapi import FastAPI
from pydantic import BaseModel
import threading
import uvicorn

app = FastAPI()


class StoryRequest(BaseModel):
    prompt: str


@app.post("/generate")
def generate_story(request: StoryRequest):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": request.prompt
        }
    ]

    story = chat_stream(
        llm,
        messages,
        max_tokens=2048
    )

    return {
        "story": story
    }

## 5. Chat helper

This wraps `create_chat_completion`, streams tokens as they're generated, and strips `<think>...</think>` reasoning blocks from what gets displayed so you see clean, direct answers. Sampling parameters follow Qwen's recommended **instruct (non-thinking) mode** settings.

In [21]:
import re

def build_story_prompt(event, age, goal, character, language):
    return f"""
Create a children's story using the following information.

Child's age: {age}

Everyday event:
{event}

Story goal:
{goal}

Main character:
{character}

Language:
{language}

Requirements:

AGE AND LANGUAGE:
- Adapt the story strictly to the child's age.
- Use simple, everyday words that a child of this age can understand.
- Use short, simple sentences.
- Avoid advanced vocabulary, abstract ideas, metaphors, idioms,
  complicated descriptions, and long sentences.
- Use simple emotions such as happy, sad, scared, nervous, angry,
  excited, and proud.
- Prefer dialogue and simple actions over long explanations.
- For very young children, keep the story short and easy to follow.

STORY:
- Create a short, creative title.
- Stay faithful to the event and its original setting.
- Do not change the event into a significantly different situation.
- Do not introduce unnecessary danger, violence, trauma, or frightening
  situations.
- Show the story goal through the character's actions and experiences.
- Do not directly explain a moral or lesson.
- Include supportive adults or friends when appropriate.
- Give the story a clear beginning, middle, and gentle ending.
- Make the story enjoyable when read aloud by a parent.
- Keep the emotional resolution realistic and gentle.

IMPORTANT:
The story should sound like a story written FOR the child,
not like a story written ABOUT the child by an adult.

Write only the title and story.

Format:

Title: [Short story title]

[Story]
"""

def chat_stream(llm, messages, max_tokens=2048):
    """Streams a reply, hides <think>...</think> content, prints/returns the rest."""
    stream = llm.create_chat_completion(
    messages=messages,
    max_tokens=max_tokens,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    repeat_penalty=1.0,
    presence_penalty=1.5,
    stream=True,
)

    full_text = ""
    in_think = False
    visible_buffer = ""

    for chunk in stream:
        delta = chunk["choices"][0]["delta"].get("content", "")
        if not delta:
            continue
        full_text += delta

        # Track whether we're currently inside a <think> block so we don't print it.
        buf = full_text
        # Strip any complete <think>...</think> blocks and drop an unfinished trailing one.
        visible = re.sub(r"<think>.*?</think>", "", buf, flags=re.DOTALL)
        visible = re.sub(r"<think>.*$", "", visible, flags=re.DOTALL)

        new_text = visible[len(visible_buffer):]
        if new_text:
            print(new_text, end="", flush=True)
            visible_buffer = visible

    print()  # trailing newline
    # Final cleaned response (used for conversation history)
    final_reply = re.sub(r"<think>.*?</think>", "", full_text, flags=re.DOTALL).strip()
    return final_reply

## 6. Interactive chat loop

Run this cell and type your messages at the prompt. Type `exit`, `quit`, or leave the input blank to stop. Conversation history is kept for the duration of the loop so the model has context from earlier turns.

In [22]:
def build_story_prompt(event, age, goal, character, language):
    return f"""
Create a children's story using the following information.

Child's age: {age}

Everyday event:
{event}

Story goal:
{goal}

Main character:
{character}

Language:
{language}

Write a complete, engaging story appropriate for the child's age.
"""

In [23]:
age = input("Child's age: ").strip()

event = input("What happened today? ").strip()

goal = input("What should the story support? ").strip()

character = input("Choose a character: ").strip()

language = input("Language (English/Bangla): ").strip()


user_prompt = build_story_prompt(
    event=event,
    age=age,
    goal=goal,
    character=character,
    language=language
)


messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt}
]

print("\nCreating your story...\n")

story = chat_stream(llm, messages)

print("\nStory generation complete.")

Child's age: 4
What happened today? they had a swimming class and felt like they will drown but we were around them, now they're scared of going to swimming classes and nervous about it
What should the story support? courage, confidence and encouragement
Choose a character: a bear and a duck
Language (English/Bangla): english

Creating your story...

Title: Barnaby's Big Splash

Barnaby Bear loved to wiggle his paws and splash in the puddles. But today, he had a very special place to go. It was the big blue pool with the gentle water. His mom said it was swimming class!

When they arrived, Barnaby felt a tiny tummy rumble. *Gurgle, gurgle.* It wasn't hungry; it was scared. The water looked deep and wavy. "I might sink," Barnaby thought. "I might get wetter than a rain cloud!" His heart went *thump-thump-thump* like a little drum.

His best friend, Daisy Duck, waddled over. She wore her bright red swim ring. "Hello, Barnaby," she said softly. "Are you ready to float?"

Barnaby shook his

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3IMBqHDHgs6aEjkwKzknMdspbSv_4yHDxjLpNCqwM5Zmd3DzL")

In [29]:
import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

In [30]:
import requests

response = requests.get("http://127.0.0.1:8000/docs")

print(response.status_code)

INFO:     127.0.0.1:57962 - "GET /docs HTTP/1.1" 200 OK
200


In [31]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)

print("Public URL:", public_url.public_url)

Public URL: https://studied-knoll-filler.ngrok-free.dev


## Notes & troubleshooting

- **Out of memory when loading**: lower `n_ctx` in step 4 (e.g. `4096`), or switch to a smaller quant in step 3 (`UD-Q3_K_XL` or `UD-Q2_K_XL`).
- **Slow generation**: confirm step 2's sanity check printed `True` for `llama_supports_gpu_offload()` — if it printed `False`, the build didn't pick up CUDA and you're running on CPU. Re-run step 2 after restarting the runtime.
- **Session disconnects**: Colab free tier has usage limits and will disconnect idle or long-running sessions; the model download and compiled wheel aren't preserved between sessions, so re-running from the top is normal.
- **Resetting the conversation**: re-run the cell in step 6 (it resets `messages` back to just the system prompt).
- **Want the model's reasoning visible?** Remove the `chat_template_kwargs={"enable_thinking": False}` line in step 5 and skip the `<think>` stripping if you'd rather see Qwen3.5's chain-of-thought.
